In [ ]:
import os
import numpy as np
import h5py
from torch.utils.data import Dataset
import torch

class IPPGDataset(Dataset):
    def __init__(self, npy_dir, hdf5_dir, transform=None):
        """
        npy_dir: pasta dos espectrogramas .npy
        hdf5_dir: pasta dos arquivos .hdf5 com pulso
        transform: transformações a aplicar na imagem (opcional)
        """
        self.npy_dir = npy_dir
        self.hdf5_dir = hdf5_dir
        self.transform = transform
        
        # Listar todos os arquivos .npy
        self.npy_files = [f for f in os.listdir(npy_dir) if f.endswith('.npy')]
        self.npy_files.sort()  # ordena para manter consistência

    def __len__(self):
        return len(self.npy_files)

    def __getitem__(self, idx):
        npy_name = self.npy_files[idx]
        npy_path = os.path.join(self.npy_dir, npy_name)
        
        # Carregar espectrograma
        cwt_img = np.load(npy_path)  # espera shape (freqs, time)
        

        if cwt_img.shape != (60, 60):
            cwt_img = np.resize(cwt_img, (60, 60))
        
        # Normalizar (0,1)
        cwt_img = (cwt_img - np.min(cwt_img)) / (np.max(cwt_img) - np.min(cwt_img) + 1e-8)
        
        # Inserir canal (1, 60, 60)
        cwt_img = np.expand_dims(cwt_img, axis=0).astype(np.float32)

        # Carregar alvo (pulso médio 0-9s) do arquivo .hdf5 correspondente
        hdf5_name = npy_name.replace('.npy', '.hdf5')
        hdf5_path = os.path.join(self.hdf5_dir, hdf5_name)
        
        with h5py.File(hdf5_path, 'r') as f:
            pulse = f['/pulse'][:]
            time = f['/time'][:]
            
            mask = (time >= 0) & (time <= 9)
            pulse_9s = pulse[mask]
            
            if len(pulse_9s) == 0:
                pulse_mean = np.mean(pulse)
            else:
                pulse_mean = np.mean(pulse_9s)
        
        target = np.array([pulse_mean], dtype=np.float32)
        
        # Converter para tensor
        cwt_tensor = torch.tensor(cwt_img)
        target_tensor = torch.tensor(target)
        
        if self.transform:
            cwt_tensor = self.transform(cwt_tensor)
        
        return cwt_tensor, target_tensor


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class IPPGCNN(nn.Module):
    def __init__(self):
        super(IPPGCNN, self).__init__()
        
        # Conv2D: 128 filtros, kernel 3x3, stride=1, padding=1 (same padding)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        
        # Pooling: 128 filtros 2x2, stride=2, padding=1 (same padding)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)
        
        # Após pooling, calcular dimensão para flatten
        self.flatten_size = 128 * 31 * 31
        
        # Fully connected densa 32 neurônios + ReLU
        self.fc1 = nn.Linear(self.flatten_size, 32)
        
        # Saída: 1 neurônio para regressão (frequência cardíaca média)
        self.out = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        
        x = x.view(-1, self.flatten_size)  # flatten
        x = self.fc1(x)
        x = self.relu(x)
        x = self.out(x)
        return x


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from dataset import IPPGDataset
from model import IPPGCNN
import torch.optim as optim
import torch.nn as nn
import numpy as np

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Diretórios - ajustar para seus caminhos
    npy_dir = "espectrogramas_npy"
    hdf5_dir = "hdf5"
    
    # Dataset completo
    dataset = IPPGDataset(npy_dir, hdf5_dir)
    
    # Dividir treino (80%) e teste (20%)
    total_len = len(dataset)
    train_len = int(0.8 * total_len)
    test_len = total_len - train_len
    train_set, test_set = random_split(dataset, [train_len, test_len])
    
    train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=16, shuffle=False)
    
    model = IPPGCNN().to(device)
    
    criterion = nn.MSELoss()  # Erro quadrático médio
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    epochs = 30
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / train_len
        
        # Avaliação no teste
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                test_loss += loss.item() * inputs.size(0)
        test_loss /= test_len
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_loss:.4f} - Test Loss: {test_loss:.4f}")
    
    # Salvar modelo
    torch.save(model.state_dict(), "ippg_cnn_model.pth")
    print("Treinamento finalizado e modelo salvo.")

if __name__ == "__main__":
    train()
